# It Takes Time: Intensity-Aware Sequential Recommendation Benchmark

Goal: rank a battery of sequential / temporal collaborative-filtering models on
**MovieLens-1M**, **MovieLens-100K**, and Amazon Reviews 5-core (**Digital Music**, **Office Products**) under a single, reproducible methodology, and
produce results tables that compare standard SOTA baselines against three new
**IA-SASRec** (Intensity-Aware SASRec) variants that inject per-interaction
intensity (ratings / hours-played) directly into the self-attention mechanism.

**Models compared**

| Family | Models |
|---|---|
| Non-sequential baselines | `Pop`, `BPR`, `ItemKNN` |
| Sequential SOTA | `FPMC`, `GRU4Rec`, `NARM`, `SASRec`, `BERT4Rec` |
| **IA-SASRec (new)** | `IA-SASRec-Add`, `IA-SASRec-Mul`, `IA-SASRec-Val` |

**Methodology**

- Datasets: ML-1M, ML-100K, Amazon Digital Music 5-core, Amazon Office Products 5-core.
  All four use rating as intensity and ship with native unix timestamps. Steam-200k is
  excluded from sequential experiments — its CSV has no timestamps. All datasets are
  filtered to users/items with >= 5 interactions.
- Split: leave-one-out per user, sorted chronologically (`LS=valid_and_test`, `order=TO`).
- HPO: Optuna TPE + MedianPruner, `N_TRIALS=10` by default.
- Eval: full-vocabulary scoring -> HitRate, NDCG, MRR, Recall, Precision @ {10, 20, 50, 100}.
- Resumability: results persisted to `results/eval/<dataset>__<model>.json`;
  Optuna studies persist to SQLite under `results/hpo/`.

## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import optuna

import config
from data import prepare_recbole_dataset, dataset_stats
from models import MODEL_REGISTRY, list_models
from hpo import run_optuna
from runner import train_and_eval, has_result, load_result
from evaluation import aggregate_results, top_k_recommend

print('device:', config.DEVICE)
print('models:', list_models())
print('N_TRIALS:', config.N_TRIALS, '| HPO_EPOCHS:', config.HPO_EPOCHS, '| FINAL_EPOCHS:', config.FINAL_EPOCHS)

2026-05-18 18:11:57.069597: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-18 18:11:57.145856: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-18 18:11:58.837651: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


device: cuda
models: ['Pop', 'BPR', 'ItemKNN', 'FPMC', 'GRU4Rec', 'NARM', 'SASRec', 'BERT4Rec', 'IA-SASRec-Add', 'IA-SASRec-Mul', 'IA-SASRec-Val']
N_TRIALS: 25 | HPO_EPOCHS: 10 | FINAL_EPOCHS: 50


## 2. Dataset preparation

Downloads MovieLens-1M (~6 MB), MovieLens-100K (~5 MB), Amazon Digital Music
5-core (~5 MB), and Amazon Office Products 5-core (~5 MB) on first run,
converts each to RecBole atomic-file format, and caches under
`recbole_data/<dataset>/<dataset>.inter`. Subsequent runs are no-ops.

In [ ]:
DATASETS = ['ml-1m', 'ml-100k-iar', 'amazon-digital-music', 'amazon-office-products']
# Note: 'ml-100k-iar' (Intensity-Aware) — NOT plain 'ml-100k'.
# RecBole 1.2.0 hard-codes a special case for dataset name == 'ml-100k' that
# overrides our data_path to its bundled 3-column example file, silently
# dropping the intensity column. See src/config.py for context.
stats_all = {}
for ds in DATASETS:
    try:
        prepare_recbole_dataset(ds)
        stats_all[ds] = dataset_stats(ds)
    except Exception as e:
        print(f'[skip] {ds}: {e}')
pd.DataFrame(stats_all).style.format('{:.4f}')

## 3. Run benchmark (resumable)

Loops the model registry. For each model:

1. If `results/eval/<dataset>__<model>.json` exists → load it, skip training.
2. Otherwise: run Optuna HPO, then a final fit on the best params, persist JSON.

Interrupt at any point; re-running this cell continues from the next un-evaluated model.

In [3]:
MODELS_TO_RUN = list_models()  # includes IA-SASRec-Add / -Mul / -Val
DATASETS_TO_RUN = [ds for ds in DATASETS if ds in stats_all]

results = {}
for ds in DATASETS_TO_RUN:
    print(f'\n########## dataset = {ds} ##########')
    for name in MODELS_TO_RUN:
        print(f'\n=== {ds} / {name} ===')
        if has_result(ds, name):
            r = load_result(ds, name)
            results[(ds, name)] = r
            print(f'  cached -> NDCG@10={r["test_result"].get("ndcg@10"):.4f}')
            continue
        hpo_out = run_optuna(ds, name)
        print(f'  HPO best_value={hpo_out["best_value"]} params={hpo_out["best_params"]}')
        r = train_and_eval(ds, name, best_params=hpo_out['best_params'])
        results[(ds, name)] = r
        print(f'  final NDCG@10={r["test_result"].get("ndcg@10"):.4f}')

print('\nAll done.')


########## dataset = ml-1m ##########

=== ml-1m / Pop ===
  cached -> NDCG@10=0.0177

=== ml-1m / BPR ===
  cached -> NDCG@10=0.0341

=== ml-1m / ItemKNN ===
  cached -> NDCG@10=0.0372

=== ml-1m / FPMC ===
  cached -> NDCG@10=0.0813

=== ml-1m / GRU4Rec ===
  cached -> NDCG@10=0.1436

=== ml-1m / NARM ===
  cached -> NDCG@10=0.1163

=== ml-1m / SASRec ===
  cached -> NDCG@10=0.1272

=== ml-1m / BERT4Rec ===
  cached -> NDCG@10=0.1237

=== ml-1m / IA-SASRec-Add ===
  cached -> NDCG@10=0.1340

=== ml-1m / IA-SASRec-Mul ===
  cached -> NDCG@10=0.1516

=== ml-1m / IA-SASRec-Val ===
  cached -> NDCG@10=0.1411

########## dataset = ml-100k ##########

=== ml-100k / Pop ===
  cached -> NDCG@10=0.0413

=== ml-100k / BPR ===
  cached -> NDCG@10=0.0644

=== ml-100k / ItemKNN ===
  cached -> NDCG@10=0.0615

=== ml-100k / FPMC ===
  cached -> NDCG@10=0.0481

=== ml-100k / GRU4Rec ===
[hpo] GRU4Rec: running 25 new trials (already have 0).
[hpo] GRU4Rec: trial 1/25 (COMPLETE) value=0.0560
[hpo] G

[W 2026-05-18 18:07:15,356] Trial 6 failed with parameters: {'n_layers': 2, 'n_heads': 1, 'hidden_size': 64, 'inner_size': 128, 'hidden_dropout_prob': 0.2641531692142519, 'attn_dropout_prob': 0.4022204554172195, 'learning_rate': 0.0002447491657907381, 'intensity_norm': 'minmax'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/coder/.pyenv/versions/3.12.0/lib/python3.12/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/home/coder/projects/phd/rec-sys-research/it_takes_time/src/hpo.py", line 126, in <lambda>
    lambda t: _objective(dataset_name, model_name, t),
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/coder/projects/phd/rec-sys-research/it_takes_time/src/hpo.py", line 54, in _objective
    out = train_one(
          ^^^^^^^^^^
  File "/home/coder/projects/phd/rec-sys-research/it_takes_time/src/runner.py", line 133, 

KeyboardInterrupt: 

## 4. Results table

In [ ]:
key_cols = ['model', 'hit@10', 'ndcg@10', 'mrr@10', 'recall@10', 'hit@100', 'ndcg@100', 'train_seconds']
tables = {}
for ds in DATASETS_TO_RUN:
    df_ds = aggregate_results(ds)
    cols = [c for c in key_cols if c in df_ds.columns]
    tables[ds] = df_ds[cols].round(4)
    print(f'\n--- {ds} ---')
    print(tables[ds].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
ndcg_cols = ['ndcg@10', 'ndcg@20', 'ndcg@50', 'ndcg@100']
fig, axes = plt.subplots(len(tables), 1, figsize=(10, 4 * len(tables)), squeeze=False)
for ax, (ds, tbl) in zip(axes.flatten(), tables.items()):
    cols = [c for c in ndcg_cols if c in tbl.columns]
    tbl.set_index('model')[cols].plot.bar(ax=ax)
    ax.set_ylabel('NDCG'); ax.set_title(f'NDCG@K by model — {ds}')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()

## 5. Top-100 recommendations demo

Confirms the "next-1 → top-K" path: for the best model on each dataset we
score the entire item vocab at the next position, mask items already in the
user's history, and take the top 100. Models are reloaded from their
checkpoints.

In [ ]:
from pathlib import Path
import numpy as np
from recbole.utils import get_model, init_seed
from recbole.data import create_dataset, data_preparation
from runner import _build_config

topk_per_dataset = {}
for ds, tbl in tables.items():
    best_model_name = tbl.iloc[0]['model']
    print(f'\n=== {ds} | best model: {best_model_name} ===')
    rec = load_result(ds, best_model_name)
    ckpt = str(config.CHECKPOINT_DIR / Path(rec['checkpoint']).name)

    cfg, model_cls = _build_config(ds, best_model_name, rec.get('best_params') or {}, epochs=1, saved=True)
    init_seed(cfg['seed'], cfg['reproducibility'])
    dataset = create_dataset(cfg)
    train_data, valid_data, test_data = data_preparation(cfg, dataset)
    model_class = model_cls or get_model(cfg['model'])
    model = model_class(cfg, train_data._dataset).to(cfg['device'])
    model.load_state_dict(torch.load(ckpt, map_location=cfg['device'])['state_dict'])
    model.eval()

    sample_users = list(dataset.id2token(dataset.uid_field, np.arange(1, 4)))
    topk_per_dataset[ds] = top_k_recommend(model, dataset, test_data, sample_users, k=100)
    print(topk_per_dataset[ds].head(5).to_string(index=False))

## 6. Analysis

In [ ]:
rows = []
for ds, tbl in tables.items():
    is_seq = {n: (MODEL_REGISTRY[n]['type'] == 'sequential') for n in tbl['model']}
    family = tbl['model'].map(lambda n: 'sequential' if is_seq[n] else 'general')
    grouped = (
        tbl.assign(family=family)
           .groupby('family')[[c for c in ['ndcg@10', 'hit@10', 'mrr@10'] if c in tbl.columns]]
           .mean()
           .round(4)
    )
    grouped['dataset'] = ds
    rows.append(grouped.reset_index())
pd.concat(rows, ignore_index=True).set_index(['dataset', 'family'])

## 7. Adding a custom variant

For your next paper, drop a class into [`src/models/variants/`](../src/models/variants/)
and register it. Example:

```python
# src/models/variants/sasrec_plus.py
from recbole.model.sequential_recommender.sasrec import SASRec
import torch.nn as nn

class SASRecPlus(SASRec):
    def __init__(self, config, dataset):
        super().__init__(config, dataset)
        self.extra_proj = nn.Linear(self.hidden_size, self.hidden_size)
    def forward(self, item_seq, item_seq_len):
        out = super().forward(item_seq, item_seq_len)
        return out + self.extra_proj(out)  # toy modification
```

Then in [`src/models/__init__.py`](../src/models/__init__.py):

```python
from models.variants.sasrec_plus import SASRecPlus
MODEL_REGISTRY['SASRecPlus'] = {
    'class': SASRecPlus,
    'type': 'sequential',
    'search_space': sasrec_space,
    'static': _seq_static(),
}
```

Re-run section 3 — only `SASRecPlus` will train; everything else loads from disk.